In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# CODE FOR CREATING THE LINE PLOTS

RESULTS_DIR = r"your_path\SIMPLE_BN_DATASET\simple_bn_results"

SAVE_PLOT = False
PLOT_NAME = "simple_bn_scm_residual_vs_stability_clean_white.png"


cf_results = pd.read_csv(os.path.join(RESULTS_DIR, "final_cf_results.csv"))
residual_summaries = pd.read_csv(os.path.join(RESULTS_DIR, "final_residual_summaries.csv"))


# PREPARE DATA

def clean_model_name(x):
    x = str(x)
    if "Model 1" in x:
        return "Model 1"
    elif "Model 2" in x:
        return "Model 2"
    return x


cf_results["model_short"] = cf_results["model_name"].apply(clean_model_name)
residual_summaries["model_short"] = residual_summaries["model_name"].apply(clean_model_name)

cf_results["method"] = cf_results["method"].astype(str).str.lower()
residual_summaries["method"] = residual_summaries["method"].astype(str).str.lower()

cf_results["causal_weight"] = pd.to_numeric(cf_results["causal_weight"], errors="coerce")
residual_summaries["causal_weight"] = pd.to_numeric(residual_summaries["causal_weight"], errors="coerce")


cf_avg = (
    cf_results
    .groupby(["method", "causal_weight", "model_short"], dropna=False)["stability_score"]
    .mean()
    .reset_index()
)

res_avg = (
    residual_summaries
    .groupby(["method", "causal_weight", "model_short"], dropna=False)["avg_total_normalized_scm_residual"]
    .mean()
    .reset_index()
)

plot_df = cf_avg.merge(
    res_avg,
    on=["method", "causal_weight", "model_short"],
    how="inner"
)

normal_df = plot_df[plot_df["method"] == "normal"].copy()
causal_df = plot_df[plot_df["method"] == "causal"].copy()

causal_df["causal_weight"] = causal_df["causal_weight"].astype(float)
causal_df = causal_df.sort_values(["model_short", "causal_weight"])


# DISPLAY-ONLY OFFSETS / JITTER

x_col = "avg_total_normalized_scm_residual"
y_col = "stability_score"

x_span = plot_df[x_col].max() - plot_df[x_col].min()
y_span = plot_df[y_col].max() - plot_df[y_col].min()

if x_span == 0:
    x_span = 1

if y_span == 0:
    y_span = 1

model_x_shift = {
    "Model 1": -0.025 * x_span,
    "Model 2":  0.025 * x_span
}

model_y_shift = {
    "Model 1": -0.020 * y_span,
    "Model 2":  0.020 * y_span
}

unique_weights = sorted(causal_df["causal_weight"].dropna().unique())
weight_rank = {w: i for i, w in enumerate(unique_weights)}
weight_center = (len(unique_weights) - 1) / 2


def add_plot_offsets(df):
    df = df.copy()

    df["weight_rank"] = df["causal_weight"].map(weight_rank)
    df["weight_centered"] = df["weight_rank"].fillna(weight_center) - weight_center

    df["x_plot"] = (
        df[x_col]
        + df["model_short"].map(model_x_shift).fillna(0)
        + df["weight_centered"] * 0.005 * x_span
    )

    df["y_plot"] = (
        df[y_col]
        + df["model_short"].map(model_y_shift).fillna(0)
        - df["weight_centered"] * 0.004 * y_span
    )

    return df


causal_df = add_plot_offsets(causal_df)
normal_df = add_plot_offsets(normal_df)


# PLOT STYLE

plt.style.use("default")

fig, ax = plt.subplots(figsize=(13.5, 7))

fig.patch.set_facecolor("white")
ax.set_facecolor("white")

marker_map = {
    "Model 1": "o",
    "Model 2": "X"
}

weight_colors = {
    0.05: "#E41A1C",
    0.10: "#FF7F00",
    0.50: "#FFD700",
    0.70: "#4DAF4A",
    0.85: "#00CED1",
    1.00: "#377EB8",
    1.50: "#984EA3",
    2.00: "#F781BF",
    3.00: "#A65628",
    5.00: "#999999",
    7.00: "#000000"
}

normal_color = "lightgray"



for model in ["Model 1", "Model 2"]:
    df_m = causal_df[causal_df["model_short"] == model].sort_values("causal_weight")

    ax.plot(
        df_m["x_plot"],
        df_m["y_plot"],
        color="gray",
        alpha=0.35,
        linewidth=1.4,
        zorder=1
    )



for _, row in causal_df.iterrows():
    w = round(float(row["causal_weight"]), 2)

    ax.scatter(
        row["x_plot"],
        row["y_plot"],
        s=115,
        marker=marker_map.get(row["model_short"], "o"),
        color=weight_colors.get(w, "#333333"),
        edgecolors="black",
        linewidths=0.8,
        alpha=0.90,
        zorder=3
    )



# NORMAL DICE POINTS

for _, row in normal_df.iterrows():
    ax.scatter(
        row["x_plot"],
        row["y_plot"],
        s=115,
        marker=marker_map.get(row["model_short"], "o"),
        color=normal_color,
        edgecolors="black",
        linewidths=0.8,
        alpha=0.90,
        zorder=4
    )


# AXES, TITLE, GRID


ax.set_title(
    "SCM Residual vs Stability on Simple-BN Dataset (Averaged Over Seeds)",
    fontsize=18,
    weight="bold",
    pad=14
)

ax.set_xlabel("Average Normalized SCM Residual", fontsize=13)
ax.set_ylabel("Average Stability Score", fontsize=13)

ax.grid(True, alpha=0.25, linewidth=0.8)

for spine in ax.spines.values():
    spine.set_color("black")
    spine.set_linewidth(1.1)

ax.tick_params(axis="both", labelsize=11, colors="black")


# AXIS LIMITS BASED ON PLOTTED POINTS

x_parts = []
y_parts = []

if len(causal_df) > 0:
    x_parts.append(causal_df["x_plot"])
    y_parts.append(causal_df["y_plot"])

if len(normal_df) > 0:
    x_parts.append(normal_df["x_plot"])
    y_parts.append(normal_df["y_plot"])

all_x = pd.concat(x_parts)
all_y = pd.concat(y_parts)

x_pad = 0.08 * (all_x.max() - all_x.min())
y_pad = 0.10 * (all_y.max() - all_y.min())

ax.set_xlim(all_x.min() - x_pad, all_x.max() + x_pad)
ax.set_ylim(all_y.min() - y_pad, all_y.max() + y_pad)


# LEGEND 1: MODEL SHAPES

model_handles = [
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Model 1",
        markerfacecolor="gray",
        markeredgecolor="black",
        markersize=10,
        linewidth=0
    ),
    Line2D(
        [0], [0],
        marker="X",
        color="none",
        label="Model 2",
        markerfacecolor="gray",
        markeredgecolor="black",
        markersize=10,
        linewidth=0
    )
]

legend1 = ax.legend(
    handles=model_handles,
    title="Model",
    loc="upper left",
    bbox_to_anchor=(1.02, 1.00),
    borderaxespad=0,
    frameon=True,
    fontsize=11,
    title_fontsize=12
)

legend1.get_frame().set_facecolor("white")
legend1.get_frame().set_edgecolor("black")
legend1.get_frame().set_alpha(0.95)

ax.add_artist(legend1)



# LEGEND 2: CAUSAL WEIGHTS

weight_handles = []

for w in sorted(causal_df["causal_weight"].dropna().unique()):
    w_clean = round(float(w), 2)

    weight_handles.append(
        Line2D(
            [0], [0],
            marker="o",
            color="none",
            label=str(w_clean).rstrip("0").rstrip("."),
            markerfacecolor=weight_colors.get(w_clean, "#333333"),
            markeredgecolor="black",
            markersize=9,
            linewidth=0
        )
    )

weight_handles.append(
    Line2D(
        [0], [0],
        marker="o",
        color="none",
        label="Normal DiCE",
        markerfacecolor=normal_color,
        markeredgecolor="black",
        markersize=9,
        linewidth=0
    )
)

legend2 = ax.legend(
    handles=weight_handles,
    title="Causal weight",
    loc="upper left",
    bbox_to_anchor=(1.02, 0.78),
    borderaxespad=0,
    frameon=True,
    fontsize=10,
    title_fontsize=11
)

legend2.get_frame().set_facecolor("white")
legend2.get_frame().set_edgecolor("black")
legend2.get_frame().set_alpha(0.95)


# SHOW / SAVE

plt.tight_layout(rect=[0, 0, 0.78, 1])

if SAVE_PLOT:
    plt.savefig(
        os.path.join(RESULTS_DIR, PLOT_NAME),
        dpi=300,
        bbox_inches="tight"
    )

plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'your_path\\SIMPLE_BN_DATASET\\simple_bn_results\\final_cf_results.csv'

In [ ]:
# code to create the model to model robustness metrics
comparison = pd.read_csv(os.path.join(RESULTS_DIR, "final_comparison_summaries.csv"))

metrics = [
    (
        "avg_normalized_numeric_magnitude_between_models",
        "Normalized numeric CF distance",
        "Normalized distance between Model 1 and Model 2 CFs"
    ),
    (
        "avg_n_features_different_between_models",
        "Number of different features",
        "Feature disagreement between Model 1 and Model 2 CFs"
    )
]

summary = (
    comparison
    .groupby(["method", "causal_weight"], dropna=False)[[m[0] for m in metrics]]
    .agg(["mean", "std"])
    .reset_index()
)

summary.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in summary.columns
]

normal = summary[summary["method"] == "normal"]
causal = summary[summary["method"] == "causal"].copy()
causal = causal.sort_values("causal_weight")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (metric, ylabel, title) in zip(axes, metrics):

    mean_col = metric + "_mean"
    std_col = metric + "_std"

    ax.errorbar(
        causal["causal_weight"],
        causal[mean_col],
        yerr=causal[std_col],
        marker="o",
        linewidth=2,
        capsize=4,
        label="CausalDiCE"
    )

    ax.fill_between(
        causal["causal_weight"],
        causal[mean_col] - causal[std_col],
        causal[mean_col] + causal[std_col],
        alpha=0.15
    )

    if len(normal) > 0:
        ax.axhline(
            normal[mean_col].iloc[0],
            linestyle="--",
            linewidth=2,
            label="Normal DiCE baseline"
        )

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Causal weight")
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

axes[0].legend(loc="best")

fig.suptitle(
    "Model-to-model counterfactual stability under retraining - Simple-BN",
    fontsize=15,
    fontweight="bold",
    y=1.03
)

plt.tight_layout()
plt.show()